# Regression Models Experiment
## Comprehensive Implementation of Linear, Logistic, Multiple, Lasso, and Ridge Regression

### Objective
This experiment demonstrates various regression techniques on real-world datasets:
1. **Linear Regression** - Simple relationship modeling
2. **Multiple Linear Regression** - Multi-feature prediction
3. **Ridge Regression** - L2 regularized regression
4. **Lasso Regression** - L1 regularized regression with feature selection
5. **Logistic Regression** - Binary classification

### Datasets Used
- **Insurance Dataset**: Predicting medical insurance charges
- **Bank Dataset**: Predicting term deposit subscription

## 1. Import Required Libraries

In [ ]:
# Essential libraries for data manipulation and modeling
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score, 
                           accuracy_score, precision_score, recall_score, f1_score, 
                           classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('default')
%matplotlib inline

## 2. Load and Explore Insurance Dataset

The insurance dataset contains information about medical insurance charges based on various factors.

In [ ]:
# Load the insurance dataset
insurance_df = pd.read_csv('Exp1_2/archive/insurance.csv')

# Display basic information about the dataset
print("Insurance Dataset Overview")
print("=" * 40)
print(f"Shape: {insurance_df.shape}")
print(f"Columns: {insurance_df.columns.tolist()}")
print("\nFirst 5 rows:")
display(insurance_df.head())

print("\nDataset Info:")
print(insurance_df.info())

print("\nMissing Values:")
print(insurance_df.isnull().sum())

print("\nStatistical Summary:")
display(insurance_df.describe())

print("\nUnique values in categorical columns:")
categorical_cols = insurance_df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"{col}: {insurance_df[col].unique()}")

## 3. Data Preprocessing for Insurance Data

Converting categorical variables and preparing the data for regression analysis.

In [ ]:
# Create a copy for preprocessing
insurance_processed = insurance_df.copy()

# Encode categorical variables
# Binary encoding for sex and smoker
insurance_processed['sex'] = insurance_processed['sex'].map({'male': 1, 'female': 0})
insurance_processed['smoker'] = insurance_processed['smoker'].map({'yes': 1, 'no': 0})

# One-hot encoding for region
region_dummies = pd.get_dummies(insurance_processed['region'], prefix='region')
insurance_processed = pd.concat([insurance_processed.drop('region', axis=1), region_dummies], axis=1)

print("Processed Dataset:")
print(insurance_processed.head())

# Visualize the target variable distribution
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(insurance_processed['expenses'], bins=30, alpha=0.7, color='skyblue')
plt.title('Distribution of Insurance Expenses')
plt.xlabel('Expenses')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.boxplot(insurance_processed['expenses'])
plt.title('Box Plot of Insurance Expenses')
plt.ylabel('Expenses')
plt.tight_layout()
plt.show()

# Correlation analysis
plt.figure(figsize=(10, 8))
correlation_matrix = insurance_processed.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix - Insurance Dataset')
plt.tight_layout()
plt.show()

## 4. Linear Regression Implementation

**Linear Regression** finds the best-fit line through data points to predict a continuous target variable.
**Formula:** y = β₀ + β₁x + ε

We'll start with a simple linear regression using age to predict expenses.

In [ ]:
# Simple Linear Regression: Age vs Expenses
X_simple = insurance_processed[['age']].values
y = insurance_processed['expenses'].values

# Split the data
X_train_simple, X_test_simple, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

# Create and train the model
linear_reg = LinearRegression()
linear_reg.fit(X_train_simple, y_train)

# Make predictions
y_pred_simple = linear_reg.predict(X_test_simple)

# Calculate metrics
r2_simple = r2_score(y_test, y_pred_simple)
rmse_simple = np.sqrt(mean_squared_error(y_test, y_pred_simple))
mae_simple = mean_absolute_error(y_test, y_pred_simple)

print("=" * 50)
print("SIMPLE LINEAR REGRESSION RESULTS")
print("=" * 50)
print(f"R² Score: {r2_simple:.4f}")
print(f"RMSE: ${rmse_simple:.2f}")
print(f"MAE: ${mae_simple:.2f}")
print(f"Coefficient (slope): {linear_reg.coef_[0]:.2f}")
print(f"Intercept: {linear_reg.intercept_:.2f}")
print(f"Equation: expenses = {linear_reg.intercept_:.2f} + {linear_reg.coef_[0]:.2f} * age")

# Visualize the results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_test_simple, y_test, alpha=0.6, color='blue', label='Actual')
plt.plot(X_test_simple, y_pred_simple, color='red', linewidth=2, label='Predicted')
plt.xlabel('Age')
plt.ylabel('Expenses')
plt.title('Linear Regression: Age vs Expenses')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_simple, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Actual Expenses')
plt.ylabel('Predicted Expenses')
plt.title('Actual vs Predicted Values')
plt.tight_layout()
plt.show()

## 5. Multiple Linear Regression Implementation

**Multiple Linear Regression** uses multiple features to predict the target variable.
**Formula:** y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ + ε

This model considers all available features to predict insurance expenses.

In [ ]:
# Multiple Linear Regression: All features
X_multi = insurance_processed.drop('expenses', axis=1).values
feature_names = insurance_processed.drop('expenses', axis=1).columns.tolist()

# Split the data
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

# Feature scaling (important for regularized models later)
scaler = StandardScaler()
X_train_multi_scaled = scaler.fit_transform(X_train_multi)
X_test_multi_scaled = scaler.transform(X_test_multi)

# Create and train the model
multi_reg = LinearRegression()
multi_reg.fit(X_train_multi_scaled, y_train_multi)

# Make predictions
y_pred_multi = multi_reg.predict(X_test_multi_scaled)

# Calculate metrics
r2_multi = r2_score(y_test_multi, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y_test_multi, y_pred_multi))
mae_multi = mean_absolute_error(y_test_multi, y_pred_multi)

print("=" * 50)
print("MULTIPLE LINEAR REGRESSION RESULTS")
print("=" * 50)
print(f"R² Score: {r2_multi:.4f}")
print(f"RMSE: ${rmse_multi:.2f}")
print(f"MAE: ${mae_multi:.2f}")
print("\\nFeature Coefficients:")
for feature, coef in zip(feature_names, multi_reg.coef_):
    print(f"{feature}: {coef:.2f}")
print(f"Intercept: {multi_reg.intercept_:.2f}")

# Feature importance visualization
plt.figure(figsize=(10, 6))
feature_importance = abs(multi_reg.coef_)
sorted_idx = np.argsort(feature_importance)[::-1]

plt.bar(range(len(feature_importance)), feature_importance[sorted_idx])
plt.xticks(range(len(feature_importance)), [feature_names[i] for i in sorted_idx], rotation=45)
plt.title('Feature Importance (Absolute Coefficients) - Multiple Linear Regression')
plt.ylabel('Absolute Coefficient Value')
plt.tight_layout()
plt.show()

# Model performance visualization
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test_multi, y_pred_multi, alpha=0.6)
plt.plot([y_test_multi.min(), y_test_multi.max()], [y_test_multi.min(), y_test_multi.max()], 'r--', linewidth=2)
plt.xlabel('Actual Expenses')
plt.ylabel('Predicted Expenses')
plt.title('Multiple Linear Regression: Actual vs Predicted')

plt.subplot(1, 2, 2)
residuals = y_test_multi - y_pred_multi
plt.scatter(y_pred_multi, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Expenses')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.tight_layout()
plt.show()

## 6. Ridge Regression Implementation

**Ridge Regression** adds L2 regularization to prevent overfitting by penalizing large coefficients.
**Formula:** Cost = MSE + α∑(βᵢ²) where α is the regularization parameter.

Ridge regression shrinks coefficients toward zero but never makes them exactly zero.

In [ ]:
# Ridge Regression with cross-validation to find optimal alpha
alphas = [0.1, 1, 10, 100, 1000]
ridge_scores = []

# Test different alpha values
for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    scores = cross_val_score(ridge, X_train_multi_scaled, y_train_multi, cv=5, scoring='r2')
    ridge_scores.append(scores.mean())
    print(f"Alpha: {alpha}, Cross-validation R² Score: {scores.mean():.4f}")

# Find best alpha
best_alpha = alphas[np.argmax(ridge_scores)]
print(f"\\nBest alpha: {best_alpha}")

# Train Ridge regression with best alpha
ridge_reg = Ridge(alpha=best_alpha)
ridge_reg.fit(X_train_multi_scaled, y_train_multi)

# Make predictions
y_pred_ridge = ridge_reg.predict(X_test_multi_scaled)

# Calculate metrics
r2_ridge = r2_score(y_test_multi, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test_multi, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test_multi, y_pred_ridge)

print("\\n" + "=" * 50)
print("RIDGE REGRESSION RESULTS")
print("=" * 50)
print(f"Best Alpha: {best_alpha}")
print(f"R² Score: {r2_ridge:.4f}")
print(f"RMSE: ${rmse_ridge:.2f}")
print(f"MAE: ${mae_ridge:.2f}")

# Compare coefficients between Linear and Ridge regression
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(alphas, ridge_scores, 'bo-')
plt.xlabel('Alpha (Regularization strength)')
plt.ylabel('Cross-validation R² Score')
plt.title('Ridge Regression: Alpha vs Performance')
plt.xscale('log')

plt.subplot(2, 2, 2)
width = 0.35
x = np.arange(len(feature_names))
plt.bar(x - width/2, multi_reg.coef_, width, label='Linear Regression', alpha=0.7)
plt.bar(x + width/2, ridge_reg.coef_, width, label='Ridge Regression', alpha=0.7)
plt.xlabel('Features')
plt.ylabel('Coefficient Value')
plt.title('Coefficient Comparison: Linear vs Ridge')
plt.xticks(x, feature_names, rotation=45)
plt.legend()

plt.subplot(2, 2, 3)
plt.scatter(y_test_multi, y_pred_ridge, alpha=0.6)
plt.plot([y_test_multi.min(), y_test_multi.max()], [y_test_multi.min(), y_test_multi.max()], 'r--', linewidth=2)
plt.xlabel('Actual Expenses')
plt.ylabel('Predicted Expenses')
plt.title('Ridge Regression: Actual vs Predicted')

plt.subplot(2, 2, 4)
ridge_residuals = y_test_multi - y_pred_ridge
plt.scatter(y_pred_ridge, ridge_residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Expenses')
plt.ylabel('Residuals')
plt.title('Ridge Regression: Residual Plot')

plt.tight_layout()
plt.show()

## 7. Lasso Regression Implementation

**Lasso Regression** uses L1 regularization, which can shrink coefficients to exactly zero, performing feature selection.
**Formula:** Cost = MSE + α∑|βᵢ| where α is the regularization parameter.

Lasso is particularly useful for feature selection and creating sparse models.

In [ ]:
# Lasso Regression with cross-validation to find optimal alpha
lasso_scores = []

# Test different alpha values
for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=2000)
    scores = cross_val_score(lasso, X_train_multi_scaled, y_train_multi, cv=5, scoring='r2')
    lasso_scores.append(scores.mean())
    print(f"Alpha: {alpha}, Cross-validation R² Score: {scores.mean():.4f}")

# Find best alpha
best_alpha_lasso = alphas[np.argmax(lasso_scores)]
print(f"\\nBest alpha for Lasso: {best_alpha_lasso}")

# Train Lasso regression with best alpha
lasso_reg = Lasso(alpha=best_alpha_lasso, max_iter=2000)
lasso_reg.fit(X_train_multi_scaled, y_train_multi)

# Make predictions
y_pred_lasso = lasso_reg.predict(X_test_multi_scaled)

# Calculate metrics
r2_lasso = r2_score(y_test_multi, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test_multi, y_pred_lasso))
mae_lasso = mean_absolute_error(y_test_multi, y_pred_lasso)

print("\\n" + "=" * 50)
print("LASSO REGRESSION RESULTS")
print("=" * 50)
print(f"Best Alpha: {best_alpha_lasso}")
print(f"R² Score: {r2_lasso:.4f}")
print(f"RMSE: ${rmse_lasso:.2f}")
print(f"MAE: ${mae_lasso:.2f}")

# Feature selection analysis
print("\\nFeature Selection Results:")
selected_features = []
for feature, coef in zip(feature_names, lasso_reg.coef_):
    if coef != 0:
        selected_features.append(feature)
        print(f"{feature}: {coef:.4f}")
    else:
        print(f"{feature}: {coef:.4f} (ELIMINATED)")

print(f"\\nNumber of features selected: {len(selected_features)} out of {len(feature_names)}")

# Visualization
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.plot(alphas, lasso_scores, 'go-', label='Lasso')
plt.plot(alphas, ridge_scores, 'bo-', label='Ridge')
plt.xlabel('Alpha (Regularization strength)')
plt.ylabel('Cross-validation R² Score')
plt.title('Lasso vs Ridge: Alpha vs Performance')
plt.xscale('log')
plt.legend()

plt.subplot(2, 3, 2)
width = 0.25
x = np.arange(len(feature_names))
plt.bar(x - width, multi_reg.coef_, width, label='Linear', alpha=0.7)
plt.bar(x, ridge_reg.coef_, width, label='Ridge', alpha=0.7)
plt.bar(x + width, lasso_reg.coef_, width, label='Lasso', alpha=0.7)
plt.xlabel('Features')
plt.ylabel('Coefficient Value')
plt.title('Coefficient Comparison: Linear vs Ridge vs Lasso')
plt.xticks(x, feature_names, rotation=45)
plt.legend()

plt.subplot(2, 3, 3)
non_zero_idx = lasso_reg.coef_ != 0
plt.bar(np.array(feature_names)[non_zero_idx], lasso_reg.coef_[non_zero_idx])
plt.title('Selected Features (Lasso)')
plt.ylabel('Coefficient Value')
plt.xticks(rotation=45)

plt.subplot(2, 3, 4)
plt.scatter(y_test_multi, y_pred_lasso, alpha=0.6)
plt.plot([y_test_multi.min(), y_test_multi.max()], [y_test_multi.min(), y_test_multi.max()], 'r--', linewidth=2)
plt.xlabel('Actual Expenses')
plt.ylabel('Predicted Expenses')
plt.title('Lasso Regression: Actual vs Predicted')

plt.subplot(2, 3, 5)
lasso_residuals = y_test_multi - y_pred_lasso
plt.scatter(y_pred_lasso, lasso_residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Expenses')
plt.ylabel('Residuals')
plt.title('Lasso Regression: Residual Plot')

plt.subplot(2, 3, 6)
# Feature importance (absolute coefficients)
feature_importance_lasso = abs(lasso_reg.coef_)
sorted_idx = np.argsort(feature_importance_lasso)[::-1]
plt.bar(range(len(feature_importance_lasso)), feature_importance_lasso[sorted_idx])
plt.xticks(range(len(feature_importance_lasso)), [feature_names[i] for i in sorted_idx], rotation=45)
plt.title('Feature Importance - Lasso Regression')
plt.ylabel('Absolute Coefficient Value')

plt.tight_layout()
plt.show()

## 8. Load and Explore Bank Dataset

The bank dataset contains information about marketing campaign data for term deposit subscriptions. This is perfect for demonstrating **Logistic Regression** for binary classification.

In [ ]:
# Load the bank dataset
bank_df = pd.read_csv('Exp1_2/archive (1)/bank.csv', delimiter=';')

print("Bank Dataset Overview")
print("=" * 40)
print(f"Shape: {bank_df.shape}")
print(f"Columns: {bank_df.columns.tolist()}")
print("\\nFirst 5 rows:")
display(bank_df.head())

print("\\nDataset Info:")
print(bank_df.info())

print("\\nMissing Values:")
print(bank_df.isnull().sum())

print("\\nTarget variable distribution:")
print(bank_df['y'].value_counts())
print("\\nTarget variable percentage:")
print(bank_df['y'].value_counts(normalize=True) * 100)

# Visualize target distribution
plt.figure(figsize=(12, 8))

plt.subplot(2, 3, 1)
bank_df['y'].value_counts().plot(kind='bar')
plt.title('Target Distribution (y)')
plt.ylabel('Count')

plt.subplot(2, 3, 2)
bank_df['age'].hist(bins=30, alpha=0.7)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')

plt.subplot(2, 3, 3)
bank_df['balance'].hist(bins=30, alpha=0.7)
plt.title('Balance Distribution')
plt.xlabel('Balance')
plt.ylabel('Frequency')

plt.subplot(2, 3, 4)
pd.crosstab(bank_df['job'], bank_df['y']).plot(kind='bar', stacked=True)
plt.title('Job vs Target')
plt.xticks(rotation=45)
plt.ylabel('Count')

plt.subplot(2, 3, 5)
pd.crosstab(bank_df['marital'], bank_df['y']).plot(kind='bar')
plt.title('Marital Status vs Target')
plt.ylabel('Count')

plt.subplot(2, 3, 6)
pd.crosstab(bank_df['education'], bank_df['y']).plot(kind='bar')
plt.title('Education vs Target')
plt.xticks(rotation=45)
plt.ylabel('Count')

plt.tight_layout()
plt.show()

## 9. Data Preprocessing for Bank Data

Preparing the bank dataset for logistic regression by encoding categorical variables and handling the target variable.

In [ ]:
# Create a copy for preprocessing
bank_processed = bank_df.copy()

# Encode target variable
bank_processed['y'] = bank_processed['y'].map({'yes': 1, 'no': 0})

# Select relevant features for logistic regression (focusing on key numerical and categorical features)
# We'll use a subset of features to avoid overfitting and for better interpretation
selected_features = ['age', 'balance', 'duration', 'campaign', 'previous', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']

bank_subset = bank_processed[selected_features + ['y']].copy()

# Encode categorical variables using Label Encoding for simplicity
label_encoders = {}
categorical_columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']

for col in categorical_columns:
    if col in bank_subset.columns:
        le = LabelEncoder()
        bank_subset[col] = le.fit_transform(bank_subset[col])
        label_encoders[col] = le

print("Processed Bank Dataset:")
print(bank_subset.head())
print(f"\\nDataset shape: {bank_subset.shape}")
print(f"Features: {bank_subset.columns[:-1].tolist()}")
print(f"Target distribution: {bank_subset['y'].value_counts().to_dict()}")

# Prepare features and target
X_bank = bank_subset.drop('y', axis=1)
y_bank = bank_subset['y']

# Split the data
X_train_bank, X_test_bank, y_train_bank, y_test_bank = train_test_split(
    X_bank, y_bank, test_size=0.2, random_state=42, stratify=y_bank
)

# Scale the features
scaler_bank = StandardScaler()
X_train_bank_scaled = scaler_bank.fit_transform(X_train_bank)
X_test_bank_scaled = scaler_bank.transform(X_test_bank)

print(f"\\nTraining set shape: {X_train_bank_scaled.shape}")
print(f"Test set shape: {X_test_bank_scaled.shape}")
print(f"Training set target distribution: {pd.Series(y_train_bank).value_counts().to_dict()}")

## 10. Logistic Regression Implementation

**Logistic Regression** is used for binary classification. It uses the logistic function (sigmoid) to model the probability.
**Formula:** P(y=1) = 1 / (1 + e^(-(β₀ + β₁x₁ + ... + βₙxₙ)))

The model predicts whether a customer will subscribe to a term deposit (yes/no).

In [ ]:
# Logistic Regression Implementation
logistic_reg = LogisticRegression(random_state=42, max_iter=1000)
logistic_reg.fit(X_train_bank_scaled, y_train_bank)

# Make predictions
y_pred_bank = logistic_reg.predict(X_test_bank_scaled)
y_pred_proba_bank = logistic_reg.predict_proba(X_test_bank_scaled)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test_bank, y_pred_bank)
precision = precision_score(y_test_bank, y_pred_bank)
recall = recall_score(y_test_bank, y_pred_bank)
f1 = f1_score(y_test_bank, y_pred_bank)

print("=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\\nClassification Report:")
print(classification_report(y_test_bank, y_pred_bank))

# Feature importance (coefficients)
feature_names_bank = X_bank.columns.tolist()
print("\\nFeature Coefficients (Log Odds):")
for feature, coef in zip(feature_names_bank, logistic_reg.coef_[0]):
    print(f"{feature}: {coef:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test_bank, y_pred_bank)
print(f"\\nConfusion Matrix:")
print(cm)

# Visualization
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')

plt.subplot(2, 3, 2)
# Feature importance (absolute coefficients)
feature_importance_log = abs(logistic_reg.coef_[0])
sorted_idx = np.argsort(feature_importance_log)[::-1]
plt.bar(range(len(feature_importance_log)), feature_importance_log[sorted_idx])
plt.xticks(range(len(feature_importance_log)), [feature_names_bank[i] for i in sorted_idx], rotation=45)
plt.title('Feature Importance (Absolute Coefficients)')
plt.ylabel('Absolute Coefficient Value')

plt.subplot(2, 3, 3)
plt.hist(y_pred_proba_bank[y_test_bank == 0], alpha=0.7, label='No (Actual)', bins=20)
plt.hist(y_pred_proba_bank[y_test_bank == 1], alpha=0.7, label='Yes (Actual)', bins=20)
plt.xlabel('Predicted Probability')
plt.ylabel('Frequency')
plt.title('Distribution of Predicted Probabilities')
plt.legend()

plt.subplot(2, 3, 4)
# ROC Curve
from sklearn.metrics import roc_curve, roc_auc_score
fpr, tpr, thresholds = roc_curve(y_test_bank, y_pred_proba_bank)
auc_score = roc_auc_score(y_test_bank, y_pred_proba_bank)
plt.plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()

plt.subplot(2, 3, 5)
# Precision-Recall Curve
from sklearn.metrics import precision_recall_curve, average_precision_score
precision_curve, recall_curve, _ = precision_recall_curve(y_test_bank, y_pred_proba_bank)
ap_score = average_precision_score(y_test_bank, y_pred_proba_bank)
plt.plot(recall_curve, precision_curve, linewidth=2, label=f'AP = {ap_score:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()

plt.subplot(2, 3, 6)
# Coefficients visualization
colors = ['red' if coef < 0 else 'blue' for coef in logistic_reg.coef_[0]]
plt.barh(range(len(feature_names_bank)), logistic_reg.coef_[0], color=colors, alpha=0.7)
plt.yticks(range(len(feature_names_bank)), feature_names_bank)
plt.xlabel('Coefficient Value')
plt.title('Logistic Regression Coefficients')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)

plt.tight_layout()
plt.show()

print(f"\\nAUC Score: {auc_score:.4f}")
print(f"Average Precision Score: {ap_score:.4f}")

## 11. Model Performance Evaluation and Comparison

Let's compare the performance of all regression models and summarize our findings.

In [ ]:
# Create comprehensive comparison
print("=" * 80)
print("COMPREHENSIVE MODEL COMPARISON")
print("=" * 80)

# Regression Models Comparison (Insurance Data)
regression_results = pd.DataFrame({
    'Model': ['Linear Regression (Simple)', 'Multiple Linear', 'Ridge Regression', 'Lasso Regression'],
    'R² Score': [r2_simple, r2_multi, r2_ridge, r2_lasso],
    'RMSE': [rmse_simple, rmse_multi, rmse_ridge, rmse_lasso],
    'MAE': [mae_simple, mae_multi, mae_ridge, mae_lasso]
})

print("\\nREGRESSION MODELS (Insurance Dataset):")
print(regression_results.round(4))

# Classification Model Results (Bank Data)
print(f"\\nLOGISTIC REGRESSION (Bank Dataset):")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC Score: {auc_score:.4f}")

# Visualization of model comparison
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
models = regression_results['Model']
r2_scores = regression_results['R² Score']
bars = plt.bar(models, r2_scores, color=['skyblue', 'lightgreen', 'orange', 'pink'])
plt.title('R² Score Comparison (Higher = Better)')
plt.ylabel('R² Score')
plt.xticks(rotation=45)
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{height:.3f}', ha='center', va='bottom')

plt.subplot(2, 3, 2)
rmse_scores = regression_results['RMSE']
bars = plt.bar(models, rmse_scores, color=['skyblue', 'lightgreen', 'orange', 'pink'])
plt.title('RMSE Comparison (Lower = Better)')
plt.ylabel('RMSE')
plt.xticks(rotation=45)
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 100,
             f'{height:.0f}', ha='center', va='bottom')

plt.subplot(2, 3, 3)
mae_scores = regression_results['MAE']
bars = plt.bar(models, mae_scores, color=['skyblue', 'lightgreen', 'orange', 'pink'])
plt.title('MAE Comparison (Lower = Better)')
plt.ylabel('MAE')
plt.xticks(rotation=45)
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 50,
             f'{height:.0f}', ha='center', va='bottom')

plt.subplot(2, 3, 4)
# Feature count comparison
feature_counts = [1, len(feature_names), len(feature_names), len(selected_features)]
plt.bar(models, feature_counts, color=['skyblue', 'lightgreen', 'orange', 'pink'])
plt.title('Number of Features Used')
plt.ylabel('Feature Count')
plt.xticks(rotation=45)

plt.subplot(2, 3, 5)
# Model complexity visualization
complexity = ['Simple', 'Complex', 'Regularized', 'Sparse']
plt.bar(models, [1, 3, 2, 2], color=['skyblue', 'lightgreen', 'orange', 'pink'])
plt.title('Model Complexity (Subjective Scale)')
plt.ylabel('Complexity Level')
plt.xticks(rotation=45)

plt.subplot(2, 3, 6)
# Classification metrics for logistic regression
log_metrics = [accuracy, precision, recall, f1]
log_metric_names = ['Accuracy', 'Precision', 'Recall', 'F1']
bars = plt.bar(log_metric_names, log_metrics, color='lightcoral')
plt.title('Logistic Regression Metrics')
plt.ylabel('Score')
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Summary and Insights
print("\\n" + "=" * 80)
print("KEY INSIGHTS AND RECOMMENDATIONS")
print("=" * 80)

print("\\n📊 REGRESSION MODELS (Insurance Premium Prediction):")
print("   • Multiple Linear Regression achieved the highest R² score")
print("   • Ridge Regression provided good balance between bias and variance")
print("   • Lasso Regression performed feature selection, keeping only important features")
print("   • Simple Linear Regression serves as a good baseline")

print("\\n🎯 LOGISTIC REGRESSION (Bank Marketing Prediction):")
print("   • Model shows decent classification performance")
print("   • Feature importance reveals key factors for term deposit subscription")
print("   • Precision and recall balance suggests model reliability")

print("\\n🔍 HOW EACH MODEL WORKS:")
print("\\n1. LINEAR REGRESSION:")
print("   - Finds best-fit line through data points")
print("   - Minimizes sum of squared errors")
print("   - Fast and interpretable")

print("\\n2. MULTIPLE LINEAR REGRESSION:")
print("   - Extends linear regression to multiple features")
print("   - Can capture complex relationships")
print("   - Risk of overfitting with many features")

print("\\n3. RIDGE REGRESSION:")
print("   - Adds L2 penalty (sum of squared coefficients)")
print("   - Controls overfitting by shrinking coefficients")
print("   - Never eliminates features completely")

print("\\n4. LASSO REGRESSION:")
print("   - Adds L1 penalty (sum of absolute coefficients)")
print("   - Performs automatic feature selection")
print("   - Creates sparse models by setting coefficients to zero")

print("\\n5. LOGISTIC REGRESSION:")
print("   - Uses sigmoid function for probability estimation")
print("   - Perfect for binary classification problems")
print("   - Coefficients represent log-odds ratios")

print("\\n✅ BEST PRACTICES:")
print("   • Use cross-validation for hyperparameter tuning")
print("   • Scale features for regularized models")
print("   • Consider domain knowledge when selecting features")
print("   • Evaluate multiple metrics, not just one")